# 06 — Aprendizaje no supervisado

## Motivación

Todo el módulo hasta ahora fue **aprendizaje supervisado**: cada dataset traía un
objetivo $y$ — un precio, un diagnóstico — y el modelo aprendía a predecirlo. En
**aprendizaje no supervisado** no hay $y$: solo características $\mathbf{x}$, y la
pregunta cambia de "¿cómo predigo esto?" a "¿qué estructura tienen estos datos por sí
mismos?".

Esta sesión cubre dos preguntas de ese tipo:

1. **Reducción de dimensionalidad** (PCA): los datos tienen 64 características —
   demasiadas para graficar. ¿Se pueden proyectar a 2D sin perder la estructura
   principal?
2. **Clustering** (k-means): sin usar ninguna etiqueta, ¿el algoritmo encuentra
   grupos que coincidan con las categorías reales?

El dataset **Digits** (imágenes de 8×8 píxeles de dígitos escritos a mano) sirve para
ambas preguntas; el cierre usa **Olivetti Faces** para ver PCA aplicado a rostros —
las *eigenfaces*.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DATA_DIR = Path("../../datos")

## 1. Reducción de dimensionalidad: PCA

**Digits** son imágenes de 8×8 píxeles — cada una es un punto en $\mathbb{R}^{64}$.
No hay forma directa de graficar 64 dimensiones. El **análisis de componentes
principales** (PCA) busca las direcciones donde los datos varían más, y proyecta
sobre las primeras — la proyección que conserva la mayor cantidad de información
posible medida como varianza.

Formalmente, con $X$ centrada (media cero en cada columna), la primera componente es
la dirección $\mathbf{v}_1$ (con $\lVert\mathbf{v}_1\rVert = 1$) que maximiza la
varianza de los datos proyectados sobre ella:

$$\mathbf{v}_1 = \arg\max_{\lVert\mathbf{v}\rVert = 1} \text{Var}(X\mathbf{v})
= \arg\max_{\lVert\mathbf{v}\rVert = 1} \mathbf{v}^\top C \mathbf{v},
\qquad C = \frac{1}{n} X^\top X$$

donde $C$ es la matriz de covarianza de $X$. La solución es el eigenvector de $C$ con
el eigenvalor más grande; ese eigenvalor **es** la varianza que esa componente
explica. Las siguientes componentes son los eigenvectores siguientes, en orden
decreciente de eigenvalor, cada uno ortogonal a los anteriores.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X, y = digits.data, digits.target

X_scaled = StandardScaler().fit_transform(X)
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_scaled)

print(f"shape original: {X.shape}   shape proyectado: {X_2d.shape}")
print(f"varianza explicada por cada componente: {pca.explained_variance_ratio_.round(3)}")

In [ ]:
fig = go.Figure()
for digit in range(10):
    mask = y == digit
    fig.add_trace(go.Scatter(
        x=X_2d[mask, 0], y=X_2d[mask, 1], mode="markers", name=str(digit),
        marker=dict(size=5, opacity=0.7),
    ))
fig.update_layout(
    title="Digits proyectado a 2D con PCA (color = dígito real)",
    xaxis_title="componente principal 1",
    yaxis_title="componente principal 2",
    template="plotly_white",
    height=550,
)
fig.show()

El color viene de la etiqueta real — PCA nunca la usó, solo vio $X$. Aun así, varios
dígitos forman regiones separadas: la estructura que PCA preserva (máxima varianza)
coincide en buena parte con la estructura que distingue un dígito de otro. No es
perfecto — 2 componentes de 64 pierden información, y se ve en las zonas donde los
colores se mezclan.

### ¿Cuántas componentes hacen falta?

Dos componentes son fáciles de graficar pero descartan información. La suma
acumulada de varianza explicada, como función del número de componentes, cuantifica
cuánto se pierde en cada punto de corte.

In [ ]:
pca_full = PCA(random_state=42).fit(X_scaled)
varianza_acumulada = np.cumsum(pca_full.explained_variance_ratio_)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=np.arange(1, len(varianza_acumulada) + 1), y=varianza_acumulada,
    mode="lines+markers", marker=dict(size=4),
))
fig.add_hline(y=0.9, line_dash="dash", line_color="gray", annotation_text="90%")
fig.update_layout(
    title="Varianza explicada acumulada",
    xaxis_title="número de componentes",
    yaxis_title="varianza acumulada",
    template="plotly_white",
)
fig.show()

## 2. Clustering: k-means

PCA responde "¿cómo veo los datos?". **k-means** responde "¿qué grupos hay?", sin usar
ninguna etiqueta. Dado un número de grupos $k$, busca $k$ centroides
$\mathbf{\mu}_1, \ldots, \mathbf{\mu}_k$ y una asignación de cada punto a un centroide
que minimicen la suma de distancias cuadradas dentro de cada grupo:

$$\min_{\mathbf{\mu}_1, \ldots, \mathbf{\mu}_k} \sum_{i=1}^{n} \min_{j} \lVert \mathbf{x}_i - \mathbf{\mu}_j \rVert^2$$

El algoritmo (Lloyd) itera dos pasos hasta que las asignaciones dejan de cambiar:

1. **Asignar**: cada punto se asigna al centroide más cercano.
2. **Actualizar**: cada centroide se recalcula como el promedio de los puntos
   asignados a él.

Se aplica k-means a Digits con $k=10$ — el mismo número que dígitos reales, aunque el
algoritmo no sabe eso — y se compara contra las etiquetas verdaderas.

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

tabla = pd.crosstab(y, clusters)
tabla.index.name = "dígito real"
tabla.columns.name = "cluster"

fig = go.Figure(go.Heatmap(
    z=tabla.values, x=[f"cluster {c}" for c in tabla.columns],
    y=[str(d) for d in tabla.index],
    colorscale="Blues", text=tabla.values, texttemplate="%{text}", textfont=dict(size=10),
))
fig.update_layout(
    title="Dígito real vs. cluster asignado por k-means",
    xaxis_title="cluster", yaxis_title="dígito real",
    template="plotly_white",
    height=500,
)
fig.show()

Cada fila muestra cómo se reparten los ejemplos de un dígito entre los 10 clusters.
Un dígito "limpio" concentra casi todos sus ejemplos en una sola columna — k-means lo
aisló en un grupo propio sin haber visto la etiqueta. Los dígitos con trazos
parecidos entre sí (por ejemplo, algunos estilos de escribir 3, 8 y 9) reparten sus
ejemplos entre varios clusters: k-means agrupa por cercanía en el espacio de píxeles,
que no siempre coincide con la categoría que un humano asignaría.

## 3. Eigenfaces: PCA sobre rostros

**Olivetti Faces** contiene 400 fotografías de 40 personas (10 cada una), imágenes de
64×64 píxeles — cada una es un punto en $\mathbb{R}^{4096}$. Las componentes
principales de este dataset tienen forma de imagen — se conocen como
*eigenfaces* — y cualquier rostro del dataset se puede reconstruir, de forma
aproximada, como combinación de un número pequeño de ellas.

In [ ]:
from sklearn.datasets import fetch_olivetti_faces

faces = fetch_olivetti_faces(data_home=DATA_DIR, shuffle=True, random_state=42)
X_faces = faces.data

print(f"shape: {X_faces.shape}   (imágenes de {faces.images.shape[1]}×{faces.images.shape[2]} píxeles)")

pca_faces = PCA(n_components=150, random_state=42).fit(X_faces)
print(f"varianza explicada con 150 componentes: {pca_faces.explained_variance_ratio_.sum():.3f}")

In [ ]:
img_shape = faces.images.shape[1:]

fig = make_subplots(rows=1, cols=6, subplot_titles=["cara promedio"] + [f"componente {i+1}" for i in range(5)])
fig.add_trace(go.Heatmap(z=pca_faces.mean_.reshape(img_shape)[::-1], colorscale="gray", showscale=False), row=1, col=1)
for i in range(5):
    fig.add_trace(
        go.Heatmap(z=pca_faces.components_[i].reshape(img_shape)[::-1], colorscale="gray", showscale=False),
        row=1, col=i + 2,
    )
for col in range(1, 7):
    fig.update_xaxes(visible=False, row=1, col=col)
    fig.update_yaxes(visible=False, scaleanchor=f"x{col}" if col > 1 else "x", row=1, col=col)
fig.update_layout(
    title="La cara promedio y las primeras 5 eigenfaces",
    template="plotly_white",
    height=280,
)
fig.show()

La cara promedio es literalmente el promedio de las 400 fotografías. Cada eigenface
es una dirección de variación alrededor de ese promedio — no representa una persona,
representa un patrón (iluminación, forma de la mandíbula, presencia de anteojos...).
Un rostro cualquiera se aproxima como la cara promedio más una combinación de estas
direcciones. Con más componentes, mejor la aproximación:

In [ ]:
rostro_original = X_faces[0]
n_componentes_prueba = [10, 50, 150]

fig = make_subplots(rows=1, cols=4, subplot_titles=["original"] + [f"{n} componentes" for n in n_componentes_prueba])
fig.add_trace(go.Heatmap(z=rostro_original.reshape(img_shape)[::-1], colorscale="gray", showscale=False), row=1, col=1)
for col, n in enumerate(n_componentes_prueba, start=2):
    pca_n = PCA(n_components=n, random_state=42).fit(X_faces)
    coeficientes = pca_n.transform(rostro_original.reshape(1, -1))
    reconstruccion = pca_n.inverse_transform(coeficientes).reshape(img_shape)
    fig.add_trace(go.Heatmap(z=reconstruccion[::-1], colorscale="gray", showscale=False), row=1, col=col)
for col in range(1, 5):
    fig.update_xaxes(visible=False, row=1, col=col)
    fig.update_yaxes(visible=False, scaleanchor=f"x{col}" if col > 1 else "x", row=1, col=col)
fig.update_layout(
    title="Reconstrucción del mismo rostro con distinto número de componentes",
    template="plotly_white",
    height=280,
)
fig.show()

## Ejercicio

Trabaja en una copia de este notebook dentro de `mi-trabajo/`.

1. **PCA a 3 componentes.** Repite la proyección de Digits usando 3 componentes en
   lugar de 2 y grafica con `go.Scatter3d`. ¿Se separan mejor los dígitos que antes se
   confundían en 2D?

2. **El número de clusters.** Ejecuta k-means sobre Digits para
   $k \in \{2, 4, 6, 8, 10, 12, 15\}$ y grafica la inercia (`kmeans.inertia_`, la suma
   de distancias cuadradas dentro de cada cluster) contra $k$. ¿En qué valor de $k$ la
   curva deja de caer con fuerza? A ese punto se le llama "método del codo".

3. **Reto — clustering de rostros.** Sobre Olivetti Faces, reduce a 50 componentes con
   PCA y aplica k-means con $k=40$ (el número real de personas). Usando
   `pd.crosstab` como en la sección 2, identifica si alguna persona quedó repartida en
   más de un cluster.